In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
client = QdrantClient("localhost", port=6333)

In [6]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
embed_model_name = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(embed_model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7129.58it/s]


In [7]:
model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype="auto", trust_remote_code=True
)

# Create a generation pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 340/340 [00:03<00:00, 93.92it/s] 


In [8]:
query = "Yayaların hakları nelerdir?"

In [9]:
# 1. Encode with E5-Small (adding the required prefix)
query_text = f"query: {query}"
query_emb = embed_model.encode([query_text])[0]

In [10]:
# 2. Use 'query_points' instead of 'search'
results = client.query_points(
    collection_name="legal_docs", query=query_emb.tolist(), limit=5
).points

In [11]:
# 3. Extract text from the payload
context = "\n\n".join(r.payload["text"] for r in results)

In [12]:
# 4. Generate with Gemma
messages = [
    {"role": "system", "content": "Sen bir Türk hukuku asistanısın."},
    {"role": "user", "content": f"Bağlam:\n{context}\n\nSoru: {query}"},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
outputs = pipe(
    prompt, max_new_tokens=500, do_sample=True, temperature=0.7, return_full_text=False
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [13]:
display(Markdown(outputs[0]["generated_text"].strip()))

Yayaların hakları şunlardır:

*   **Taşit yolu bitişiğinde ve yakınında yaya yolu, banket veya alan varsa burada yürümek zorundadır.**
*   **İki yönlü karayolu üzerinde yaya yürüyebilmelidir.**
*   **Yaya yolu ayrılmış ise, yaya yollarında yürümek zorundadır.**
*   **Yaya geçitlerinde veya kavşaklarında yürümek zorundadırlar.**
*   **Yaya yollarında geçitlerde veya zorunlu hallerde taşıt yolu üzerinde yürüyebilmelidirler.**
*   **Taşıt yolu üzerinde yürümek zorundadırlar.**
*   **Yaya yollarında, geçitlerde veya zorunlu hallerde taşıt yolu üzerinde yürüyebilmelidirler.**
*   **Yaya yollarında, geçitlerde veya zorunlu hallerde taşıt yolu üzerinde yürüyebilmelidirler.**
*   **Taşit yolu üzerinde yaya yürüyebilmelidirler.**

Bu haklar, yaya güvenliğinin sağlanması ve yaya haklarının korunması amacıyla düzenlenmiştir.